In [0]:
# simulation
import os
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array


mlflow.set_registry_uri("databricks-uc")
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"
# UC temp dir (recommended for serverless/shared)
#dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
#os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"

MODEL_NAME = "lr_end_to_end_pipeline_model"
MODEL_URI = f"models:/{MODEL_NAME}@Champion"   # use Production for jobs
# If you haven't promoted yet, temporarily use:
# MODEL_URI = f"models:/{MODEL_NAME}/latest"

INPUT_TABLE = "mlops_project.lendingclub_silver"
OUTPUT_TABLE = "mlops_project.lendingclub_gold_predictions_monitor"

model = mlflow.spark.load_model(MODEL_URI)
df = spark.table(INPUT_TABLE)

scored = (
     model.transform(df)
      .withColumn("default_proba_raw", vector_to_array("probability")[1])

      # ---- synthetic drift: bump probabilities up a bit (riskier day)
      # clamp to [0,1]
      .withColumn(
          "default_proba",
          F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("default_proba_raw") * 1.15 + 0.03))
      )

      # recompute prediction from shifted proba (threshold 0.5)
      .withColumn("prediction", F.when(F.col("default_proba") >= 0.5, F.lit(1.0)).otherwise(F.lit(0.0)))

      # simulate next day
      .withColumn("prediction_ts", F.current_timestamp())
      .withColumn("score_date", F.date_add(F.current_date(), 1))

      .withColumn("model_name", F.lit(MODEL_NAME))
      .withColumn("model_alias", F.lit("Champion"))

      .drop("default_proba_raw")
)

scored.select("score_date","prediction_ts","model_name","model_alias","default_proba","prediction","label_default").show(5, truncate=False)

(scored
 .select("score_date","prediction_ts","model_name","model_alias","default_proba","prediction","probability","label_default")
 .write.format("delta")
 .mode("append")          # <-- important (keep history)
 .saveAsTable(OUTPUT_TABLE))

print("Gold rows:", spark.table(OUTPUT_TABLE).count())

/databricks/python/lib/python3.12/site-packages/databricks/sdk/errors/base.py:87: UserWarning: The 'retry_after_secs' parameter of DatabricksError is deprecated and will be removed in a future version.
  warnings.warn(


+----------+--------------------------+----------------------------+-----------+-------------------+----------+-------------+
|score_date|prediction_ts             |model_name                  |model_alias|default_proba      |prediction|label_default|
+----------+--------------------------+----------------------------+-----------+-------------------+----------+-------------+
|2026-01-09|2026-01-08 15:31:38.464092|lr_end_to_end_pipeline_model|Champion   |0.5089782705996306 |1.0       |0            |
|2026-01-09|2026-01-08 15:31:38.464092|lr_end_to_end_pipeline_model|Champion   |0.8708776560780124 |1.0       |0            |
|2026-01-09|2026-01-08 15:31:38.464092|lr_end_to_end_pipeline_model|Champion   |0.09476899028867414|0.0       |0            |
|2026-01-09|2026-01-08 15:31:38.464092|lr_end_to_end_pipeline_model|Champion   |0.4286783653103019 |0.0       |0            |
|2026-01-09|2026-01-08 15:31:38.464092|lr_end_to_end_pipeline_model|Champion   |0.6229689452883389 |1.0       |0      

In [0]:
from pyspark.sql import functions as F

GOLD_TBL = "mlops_project.lendingclub_gold_predictions_monitor"
MODEL_NAME = "lr_end_to_end_pipeline_model"
MODEL_ALIAS = "Champion"

gold = (spark.table(GOLD_TBL)
        .select("score_date","model_name","model_alias","default_proba")
        .filter(F.col("default_proba").isNotNull())
        .filter((F.col("model_name")==MODEL_NAME) & (F.col("model_alias")==MODEL_ALIAS))
)

# baseline = first day, current = last day
baseline = gold.agg(F.min("score_date").alias("d")).collect()[0]["d"]
current  = gold.agg(F.max("score_date").alias("d")).collect()[0]["d"]

base_df = gold.filter(F.col("score_date")==baseline)
curr_df = gold.filter(F.col("score_date")==current)

def add_bin(df):
    p = F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("default_proba")))
    b = F.floor(p * 10).cast("int")
    b = F.when(b==10, 9).otherwise(b)
    return df.withColumn("score_bin", b)

base_df = add_bin(base_df)
curr_df = add_bin(curr_df)

base_total = base_df.count()
curr_total = curr_df.count()

bins = spark.range(0,10).withColumnRenamed("id","score_bin")

base_dist = base_df.groupBy("score_bin").count().withColumnRenamed("count","base_cnt")
curr_dist = curr_df.groupBy("score_bin").count().withColumnRenamed("count","curr_cnt")

dist = (bins.join(base_dist,"score_bin","left")
           .join(curr_dist,"score_bin","left")
           .fillna(0, ["base_cnt","curr_cnt"])
           .withColumn("base_pct", F.col("base_cnt")/F.lit(base_total))
           .withColumn("curr_pct", F.col("curr_cnt")/F.lit(curr_total)))

eps = 1e-6
dist = (dist
    .withColumn("base_s", F.when(F.col("base_pct")<=0, F.lit(eps)).otherwise(F.col("base_pct")))
    .withColumn("curr_s", F.when(F.col("curr_pct")<=0, F.lit(eps)).otherwise(F.col("curr_pct")))
    .withColumn("psi_component", (F.col("curr_s")-F.col("base_s")) * F.log(F.col("curr_s")/F.col("base_s")))
)

psi = dist.agg(F.sum("psi_component").alias("psi")).collect()[0]["psi"]
print("Baseline:", baseline, "Current:", current, "PSI:", float(psi))


Baseline: 2026-01-08 Current: 2026-01-09 PSI: 0.20793609990892237


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

GOLD_TBL = "mlops_project.lendingclub_gold_predictions_monitor"
MODEL_NAME = "lr_end_to_end_pipeline_model"
MODEL_ALIAS = "Champion"

gold = (spark.table(GOLD_TBL)
        .filter((F.col("model_name")==MODEL_NAME) & (F.col("model_alias")==MODEL_ALIAS))
        .select("score_date","label_default","default_proba")
        .filter(F.col("label_default").isNotNull())
        .filter(F.col("default_proba").isNotNull())
)

dates = [r["score_date"] for r in gold.select("score_date").distinct().orderBy("score_date").collect()]

evaluator = BinaryClassificationEvaluator(labelCol="label_default", rawPredictionCol="prediction", metricName="areaUnderROC")


auc_by_day = []
for d in dates:
    auc = evaluator.evaluate(
        spark.table(GOLD_TBL).filter(F.col("score_date")==d)
    )
    auc_by_day.append((d, float(auc)))

auc_by_day



[(datetime.date(2026, 1, 8), 0.6796026976063514),
 (datetime.date(2026, 1, 9), 0.6668230694948583)]

In [0]:
from pyspark.sql import functions as F

bands = (spark.table(GOLD_TBL)
  .withColumn("score_band", F.concat(F.format_number(F.floor(F.col("default_proba")*10)/10, 1), F.lit("-"),
                                    F.format_number((F.floor(F.col("default_proba")*10)+1)/10, 1)))
  .groupBy("score_date","model_name","model_alias","score_band")
  .agg(
      F.count("*").alias("n"),
      F.avg("default_proba").alias("avg_pred_proba"),
      F.avg("label_default").alias("actual_default_rate")
  )
)
bands.show(20, truncate=False)



+----------+----------------------------+-----------+----------+------+-------------------+--------------------+
|score_date|model_name                  |model_alias|score_band|n     |avg_pred_proba     |actual_default_rate |
+----------+----------------------------+-----------+----------+------+-------------------+--------------------+
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.7-0.8   |84141 |0.7432979629902808 |0.4485090502846413  |
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.6-0.7   |149050|0.6463702248451583 |0.33883931566588393 |
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.0-0.1   |42133 |0.07209745913325337|0.023141005862388153|
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.4-0.5   |236764|0.4496768981566759 |0.18712726596948862 |
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.1-0.2   |131358|0.1554123085106708 |0.04944502809117069 |
|2026-01-08|lr_end_to_end_pipeline_model|Champion   |0.9-1.0   |35591 |0.9951417579247708 |0.973

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

rows = [Row(score_date=d, auc=float(a), model_name=MODEL_NAME, model_alias=MODEL_ALIAS) for d,a in auc_by_day]
spark.createDataFrame(rows).write.format("delta").mode("append").saveAsTable("mlops_project.monitoring_auc_daily")


In [0]:
spark.createDataFrame([Row(
    baseline_date=baseline,
    score_date=current,
    psi_default_proba=float(psi),
    model_name=MODEL_NAME,
    model_alias=MODEL_ALIAS
)]).write.format("delta").mode("append").saveAsTable("mlops_project.monitoring_drift_daily")


In [0]:
(bands
 .withColumn("model_name", F.lit(MODEL_NAME))
 .withColumn("model_alias", F.lit(MODEL_ALIAS))
 .write.format("delta").mode("append")
 .saveAsTable("mlops_project.monitoring_score_bands_daily"))
